# NMT (Transformer + BPE + Noam LR + Weight Tying) Model: English to Bengali/Hindi

This notebook implements a full sequence-to-sequence (Seq2Seq) pipeline for neural machine translation (NMT) from scratch in PyTorch.

**Key Features:**
* **Architecture:** This model uses the **Transformer** architecture built from scratch.
* **Tokenization:** Uses a **custom from-scratch BPE tokenizer** instead of NLTK.
* **Optimizations:**
    1.  **Weight Tying:** The decoder's embedding and final output layers share weights.
    2.  **Custom LR Scheduler:** Uses the "Noam" optimizer from the "Attention Is All You Need" paper (warmup then inverse-sqrt decay).
* **Structure:** This notebook runs the **Bengali** pipeline first, followed by the **Hindi** pipeline, and then combines the results.
* **Checkpointing:** This model trains for 30 epochs and saves submissions for epochs **10, 20, and 30**.

## Step 1: Imports and Setup

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import json
import numpy as np
import pandas as pd
import re
import string
import nltk
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import time
import math
import os
import zipfile
import shutil

# Imports for BPE
from collections import Counter, defaultdict
import heapq

In [2]:
# -- Constants and Configuration --

# 1. DEFINE SPECIAL TOKENS
# Note: The BPE code uses string tokens. We will map them to integers.
# The BPE code also defines '<pad>', '<unk>', '<s>', '</s>' as RESERVED_TOKENS.
# We will use these and ensure our integer mapping is consistent.
PAD_TOKEN_STR = "<pad>"
UNK_TOKEN_STR = "<unk>"
SOS_TOKEN_STR = "<s>"
EOS_TOKEN_STR = "</s>"

PAD_token = 0
UNK_token = 1
SOS_token = 2
EOS_token = 3

# 2. SET DEVICE
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 3. SET FILE PATHS
TRAIN_FILE_PATH = "/kaggle/input/nmt-trainval-data/train_data1.json"
VAL_FILE_PATH = "/kaggle/input/nmt-trainval-data/val_data1.json"
TEST_FILE_PATH = "/kaggle/input/nmt-trainval-data/test_data1.json" # <-- NEW TEST FILE

# 4. --- NEW EPOCH SCHEDULE ---
EPOCH_CHECKPOINTS = [10, 20, 30] # <-- NEW SCHEDULE
FINAL_EPOCH = 30 # <-- NEW FINAL EPOCH

# 5. --- BPE VOCAB SIZE ---
BPE_VOCAB_SIZE = 15000


Using device: cuda


## Step 2: Data Loading (Shared Functions)

In [3]:
# Load Train data
try:
    with open(TRAIN_FILE_PATH, 'r') as file:
        train_data = json.load(file)
    print(f"Successfully loaded {TRAIN_FILE_PATH}")
except FileNotFoundError:
    print(f"ERROR: {TRAIN_FILE_PATH} not found. Please check the path.")

# Load Validation data
try:
    with open(VAL_FILE_PATH, 'r') as file:
        val_data = json.load(file)
    print(f"Successfully loaded {VAL_FILE_PATH}")
except FileNotFoundError:
    print(f"ERROR: {VAL_FILE_PATH} not found. Please check the path.")
    val_data = None

# Load Test data
try:
    with open(TEST_FILE_PATH, 'r') as file:
        test_data = json.load(file)
    print(f"Successfully loaded {TEST_FILE_PATH}")
except FileNotFoundError:
    print(f"ERROR: {TEST_FILE_PATH} not found. Please check the path.")
    test_data = None

Successfully loaded /kaggle/input/nmt-trainval-data/train_data1.json
Successfully loaded /kaggle/input/nmt-trainval-data/val_data1.json
Successfully loaded /kaggle/input/nmt-trainval-data/test_data1.json


In [4]:
# --- UTILITY FUNCTIONS ---

# 1. Maximum sequence length for padding
MAX_LENGTH = 100

# 2. Extract data from JSON
def extract_data(data, lang_pair_key, split):
    """Extracts source sentences, target sentences, and IDs from the loaded JSON data."""
    source_sentences = []
    target_sentences = []
    entry_ids = []
    
    # Check if the language pair and split exist
    if data is None:
        print(f"Error: Data for {split} is None (file not found?).")
        return [], [], []
    if lang_pair_key not in data:
        print(f"Error: Language pair '{lang_pair_key}' not in data.")
        return [], [], []
    if split not in data[lang_pair_key]:
        print(f"Error: Split '{split}' not in data for {lang_pair_key}.")
        return [], [], []
        
    data_entries = data[lang_pair_key][split]
    
    for entry_id, entry_data in data_entries.items():
        entry_ids.append(entry_id)
        source_sentences.append(entry_data["source"])
        
        # Test/Val sets do not have a 'target' key
        if "target" in entry_data:
            target_sentences.append(entry_data["target"])
        
    if not target_sentences and split == "Train":
        print(f"Warning: No target sentences found for {split} split.")
        return source_sentences, None, entry_ids
        
    if split != "Train":
        return source_sentences, None, entry_ids
        
    return source_sentences, target_sentences, entry_ids

# 3. Format epoch time
def epoch_time(start_time, end_time):
    """Helper function to format epoch time."""
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

print(f"MAX_LENGTH set to: {MAX_LENGTH}")
print("Utility functions defined: extract_data(), epoch_time()")


MAX_LENGTH set to: 100
Utility functions defined: extract_data(), epoch_time()


## Step 3: BPE and Preprocessing (Shared Functions)

This section contains all the shared code for BPE tokenization, preprocessing, and model architecture.

In [5]:
# --- BPE CODE (as provided) --- 
# NOTE: word end token is 256 here and whitesapce is '_'
RESERVED_TOKENS = ['<pad>', '<unk>', '<s>', '</s>']
EOW_MARKER = '256'

# defining the custom data structures used
class Node:
    """double linked list node (represents a token in a word)"""
    def __init__(self, token_index):
        self.token_index = token_index
        self.prev = None
        self.next = None
        
class SplitWord:
    """double linked list representing word's current split"""
    def __init__(self, tokens):
        self.head = None
        self.tail = None
        self.nodes = []
        prev = None
        for token in tokens:
            node = Node(token)
            node.prev = prev
            if prev:
                prev.next = node
            else:
                self.head = node
            prev = node
            self.nodes.append(node)
        self.tail = prev
    
    def tokens_list(self):
        """returns list of tokens"""
        toktokitoki = []
        node = self.head
        while node:
            toktokitoki.append(node.token_index)
            node = node.next
        return toktokitoki
    
    def merge_tokens(self, left, new_token_index):
        """merge left node with its next node, creating a new token"""
        right = left.next
        if not right:
            return None
        
        # create new merged node
        new_node = Node(new_token_index)
        new_node.prev = left.prev
        new_node.next = right.next
        
        if left.prev:
            left.prev.next = new_node
        else:
            self.head = new_node
        
        if right.next:
            right.next.prev = new_node
        else:
            self.tail = new_node
        
        # remove old nodes
        left.next = None; left.prev = None
        right.prev = None; right.next = None

        return new_node
              
def train_bpe_tokenizer(text, vocab_size):
    """trains bpe using linked list and heap"""
    print(f"Starting BPE training with vocab size {vocab_size}...")
    words = []
    for word in re.findall(r'\S+|\s+', text):
        if not word.isspace():
            words.append(word)
    words_string = []
    for word in words:
        words_string.append(word)
    
    print(f"BPE: Prepared {len(words_string)} words for training.")
    
    # encoding each word as a list of bytes + word end (256)
    corpus = []
    for word in words_string:
        encoded_word_list = SplitWord(list(word.encode('utf-8')) + [256])
        corpus.append(encoded_word_list)
    
    # initial vocab: reserved and bytes and EOW
    initial_vocab = RESERVED_TOKENS + [str(i) for i in range(257)]
    
    # count and map bigrams
    bigram_frequencies = defaultdict(int)
    bigram_positions = defaultdict(set)
    
    print("BPE: Counting initial bigrams...")
    for word_index, word in enumerate(corpus):
        node = word.head
        while node and node.next:
            bigram = (str(node.token_index),str(node.next.token_index))
            bigram_frequencies[bigram] += 1
            bigram_positions[bigram].add((word_index, node))
            node = node.next
            
    # heap to get bigram that is most 
    heap = []
    merge_index_count = 0
    for bigram, frequency in bigram_frequencies.items():
        heapq.heappush(heap, (-frequency, merge_index_count, bigram))
        merge_index_count += 1

    current_counts = dict(bigram_frequencies)
    
    merges = []
    merge_rules = {} # bigram -> new_token
    
    # helper function (updates bigram frequencies after merge)
    def update_bigram_frequency_after_merge(word_index, merge_node):
        left = merge_node.prev
        right = merge_node.next
        
        if right:
            old_bigram = (str(merge_node.token_index), str(right.token_index))
            if old_bigram in bigram_positions:
                try:
                    bigram_positions[old_bigram].remove((word_index, merge_node))
                    bigram_frequencies[old_bigram] -= 1
                except KeyError:
                    pass
                if bigram_frequencies[old_bigram] <= 0:
                    if old_bigram in bigram_frequencies: del bigram_frequencies[old_bigram]
                    if old_bigram in bigram_positions: del bigram_positions[old_bigram]
                current_counts[old_bigram] = bigram_frequencies.get(old_bigram, 0)
        
        if left:
            old_bigram = (str(left.token_index), str(merge_node.token_index))
            if old_bigram in bigram_positions:
                try:
                    bigram_positions[old_bigram].remove((word_index, left))
                    bigram_frequencies[old_bigram] -= 1
                except KeyError:
                    pass
                if bigram_frequencies[old_bigram] <= 0:
                    if old_bigram in bigram_frequencies: del bigram_frequencies[old_bigram]
                    if old_bigram in bigram_positions: del bigram_positions[old_bigram]
                current_counts[old_bigram] = bigram_frequencies.get(old_bigram, 0)
                
        if left:
            new_bigram = (str(left.token_index), str(merge_node.token_index))
            bigram_frequencies[new_bigram] = bigram_frequencies.get(new_bigram, 0) + 1
            bigram_positions[new_bigram].add((word_index, left))
            current_counts[new_bigram] = bigram_frequencies[new_bigram]
            heapq.heappush(
                heap, 
                (-bigram_frequencies[new_bigram], merge_index_count + 1000000, new_bigram)
            )
            
        if right:
            new_bigram = (str(merge_node.token_index), str(right.token_index))
            bigram_frequencies[new_bigram] = bigram_frequencies.get(new_bigram, 0) + 1
            bigram_positions[new_bigram].add((word_index, merge_node))
            current_counts[new_bigram] = bigram_frequencies[new_bigram]
            heapq.heappush(
                heap, 
                (-bigram_frequencies[new_bigram], merge_index_count + 2000000, new_bigram)
            )
    
    vocab = list(initial_vocab) # Make it a list
    
    print("BPE: Starting merge loop...")
    pbar = tqdm(total=vocab_size - len(vocab))
    while len(vocab) < vocab_size and heap:
        negative_count, _, bigram = heapq.heappop(heap)
        count = -negative_count
        
        if current_counts.get(bigram, 0) != count or count == 0:
            continue
        
        if any(tok in RESERVED_TOKENS for tok in bigram):
            continue
            
        new_token = bigram[0] + '_' + bigram[1]
        if new_token in vocab:
            continue
        
        vocab.append(new_token)
        merges.append(bigram) # Save the tuple (str, str)
        merge_rules[bigram] = new_token
        
        new_token_index = new_token 
        
        bigram_instances = list(bigram_positions[bigram])
        bigram_positions[bigram].clear()
        bigram_frequencies[bigram] = 0
        current_counts[bigram] = 0
        
        for word_index, node in bigram_instances:
            word = corpus[word_index]
            if node.next is None: # Already merged
                continue
            if (str(node.token_index), str(node.next.token_index)) != bigram:
                continue
            
            merge_node = word.merge_tokens(node, new_token_index)
            update_bigram_frequency_after_merge(word_index, merge_node)
        
        pbar.update(1)
    
    pbar.close()
    print(f"BPE: Training complete. Final vocab size: {len(vocab)}")
    
    tokenizer = {
        'vocab': set(vocab), # Use set for fast lookups
        'merges': merges,      # List of (str, str) tuples
        'merge_rules': merge_rules # Dict of (str, str) -> str
    }
    return vocab, tokenizer


def tokenise_bpe(text, tokenizer):
    # Simple whitespace regex tokenizer
    words = re.findall(r'\S+|\s+', text)
    tokens = []
    vocab = tokenizer['vocab']
    merge_rules = tokenizer['merge_rules']
    # Create a ranked map of merges
    merges_rank = {pair: i for i, pair in enumerate(tokenizer['merges'])}

    for item in words:
        if item.isspace():
            # Treat whitespace as single tokens if they are in vocab
            # (e.g., ' ') or just skip. Let's just use space.
            tokens.append(" ") # Represent space
            continue

        # Start with byte-level representation + EOW
        byte_list = list(item.encode('utf-8')) + [256]
        word_array = [str(b) for b in byte_list]

        while True:
            pairs = [(word_array[i], word_array[i + 1]) for i in range(len(word_array) - 1)]
            
            # Find the next best merge
            best_pair = None
            best_rank = float('inf')
            
            for pair in pairs:
                if pair in merge_rules and merges_rank.get(pair, float('inf')) < best_rank:
                    best_pair = pair
                    best_rank = merges_rank[pair]
            
            if best_pair is None:
                break # No more valid merges found
            
            # Apply the merge
            new_token = merge_rules[best_pair]
            new_word_array = []
            i = 0
            while i < len(word_array):
                if i < len(word_array) - 1 and (word_array[i], word_array[i+1]) == best_pair:
                    new_word_array.append(new_token)
                    i += 2
                else:
                    new_word_array.append(word_array[i])
                    i += 1
            word_array = new_word_array

        tokens.extend([token if token in vocab else UNK_TOKEN_STR for token in word_array])

    return tokens

def detokenize_bpe(tokens):
    """Detokenizes a list of BPE string tokens back to a text string."""
    
    # This reverse mapping is the core of BPE detokenization
    byte_map = {str(i): bytes([i]) for i in range(256)}
    byte_map[EOW_MARKER] = b'' # End of Word marker
    byte_map[" "] = b' ' # Handle space token

    def get_bytes(token):
        """Recursively break down a merged token (e.g., '72_101') into its base bytes."""
        if token in byte_map:
            return byte_map[token]
        if '_' in token:
            left, right = token.split('_', 1)
            return get_bytes(left) + get_bytes(right)
        return b'' # Handle <unk>, <s>, etc. by returning empty bytes
    
    full_byte_array = b''
    for token in tokens:
        full_byte_array += get_bytes(token)
    
    # Decode the full byte array, replacing errors
    return full_byte_array.decode('utf-8', errors='replace')

## Step 3.5: New BPE Vocab and Preprocessing Functions

These functions will replace the old NLTK-based `Vocab`, `preprocess_sentence`, and `encode_and_pad`.

In [6]:
class BPEVocab:
    """A simple class to hold the BPE vocab and its mappings."""
    def __init__(self, vocab_list):
        # Ensure reserved tokens are first and in the correct order
        self.vocab = list(RESERVED_TOKENS)
        for token in vocab_list:
            if token not in RESERVED_TOKENS:
                self.vocab.append(token)
        
        self.word2index = {tok: i for i, tok in enumerate(self.vocab)}
        self.index2word = {i: tok for i, tok in enumerate(self.vocab)}
        self.n_words = len(self.vocab)
        
        # Get the integer indices for special tokens from our new map
        self.pad_idx = self.word2index[PAD_TOKEN_STR]
        self.sos_idx = self.word2index[SOS_TOKEN_STR]
        self.eos_idx = self.word2index[EOS_TOKEN_STR]
        self.unk_idx = self.word2index[UNK_TOKEN_STR]
        
        # Assert they match our global constants
        assert self.pad_idx == PAD_token
        assert self.sos_idx == SOS_token
        assert self.eos_idx == EOS_token
        assert self.unk_idx == UNK_token

def bpe_preprocess_and_tokenize(sentence, tokenizer):
    """Cleans and tokenizes a sentence using BPE."""
    # BPE is case-sensitive, but we lowercase for simplicity
    sentence = sentence.lower()
    # The BPE code handles punctuation/numbers as bytes, so we just strip whitespace
    sentence = sentence.strip()
    return tokenise_bpe(sentence, tokenizer)

def encode_and_pad_bpe(sentence_tokens, bpe_vocab, max_length):
    """Converts BPE string tokens to a padded list of indices."""
    encoded = [bpe_vocab.sos_idx]
    for token in sentence_tokens:
        encoded.append(bpe_vocab.word2index.get(token, bpe_vocab.unk_idx))
    encoded.append(bpe_vocab.eos_idx)
    
    if len(encoded) > max_length:
        encoded = encoded[:max_length - 1] + [bpe_vocab.eos_idx]
        
    n_pads = max_length - len(encoded)
    encoded.extend([bpe_vocab.pad_idx] * n_pads)
    return encoded

## Step 7: Model Architecture (Transformer + Weight Tying)

This is the same Transformer architecture from before. It's compatible with our new BPE vocab.

In [7]:
class PositionalEncoding(nn.Module):
    """Injects positional information into the embeddings."""
    def __init__(self, d_model, dropout_p, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout_p)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0).transpose(0, 1) # Shape: (max_len, 1, d_model)
        self.register_buffer('pe', pe) # Register as buffer so it's not a model parameter

    def forward(self, x):
        # x shape: (seq_len, batch_size, d_model)
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention mechanism from scratch."""
    def __init__(self, d_model, n_heads, dropout_p):
        super(MultiHeadAttention, self).__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads # Dimension of each head
        
        self.fc_q = nn.Linear(d_model, d_model)
        self.fc_k = nn.Linear(d_model, d_model)
        self.fc_v = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(p=dropout_p)
        self.scale = torch.sqrt(torch.FloatTensor([self.d_k])).to(device)
        
    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]
        Q = self.fc_q(query)
        K = self.fc_k(key)
        V = self.fc_v(value)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)
        attention_weights = torch.softmax(energy, dim=-1)
        attention_weights = self.dropout(attention_weights)
        x = torch.matmul(attention_weights, V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(batch_size, -1, self.d_model)
        x = self.fc_out(x)
        return x, attention_weights

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout_p):
        super(PositionwiseFeedForward, self).__init__()
        self.fc_1 = nn.Linear(d_model, d_ff)
        self.fc_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(p=dropout_p)
    def forward(self, x):
        x = self.dropout(torch.relu(self.fc_1(x)))
        x = self.fc_2(x)
        return x

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout_p):
        super(TransformerEncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout_p)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout_p)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(p=dropout_p)
        self.dropout2 = nn.Dropout(p=dropout_p)
    def forward(self, src, src_mask):
        _src, _ = self.self_attn(src, src, src, src_mask)
        src = self.norm1(src + self.dropout1(_src))
        _src = self.ffn(src)
        src = self.norm2(src + self.dropout2(_src))
        return src

class TransformerEncoder(nn.Module):
    def __init__(self, input_dim, d_model, n_heads, d_ff, n_layers, dropout_p, max_len=100):
        super(TransformerEncoder, self).__init__()
        self.tok_embedding = nn.Embedding(input_dim, d_model, padding_idx=PAD_token)
        self.pos_embedding = PositionalEncoding(d_model, dropout_p, max_len)
        self.layers = nn.ModuleList([TransformerEncoderLayer(d_model, n_heads, d_ff, dropout_p) 
                                     for _ in range(n_layers)])
        self.dropout = nn.Dropout(p=dropout_p)
        self.scale = torch.sqrt(torch.FloatTensor([d_model])).to(device)
        
    def forward(self, src, src_mask):
        src_emb = self.tok_embedding(src) * self.scale
        src_pos = self.pos_embedding(src_emb.transpose(0, 1))
        src = self.dropout(src_pos.transpose(0, 1))
        for layer in self.layers:
            src = layer(src, src_mask)
        return src

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout_p):
        super(TransformerDecoderLayer, self).__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, n_heads, dropout_p)
        self.encoder_attn = MultiHeadAttention(d_model, n_heads, dropout_p)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout_p)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(p=dropout_p)
        self.dropout2 = nn.Dropout(p=dropout_p)
        self.dropout3 = nn.Dropout(p=dropout_p)
    def forward(self, tgt, enc_src, tgt_mask, src_mask):
        _tgt, _ = self.masked_self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = self.norm1(tgt + self.dropout1(_tgt))
        _tgt, attention = self.encoder_attn(tgt, enc_src, enc_src, src_mask)
        tgt = self.norm2(tgt + self.dropout2(_tgt))
        _tgt = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout3(_tgt))
        return tgt, attention

class TransformerDecoder(nn.Module):
    def __init__(self, output_dim, d_model, n_heads, d_ff, n_layers, dropout_p, max_len=100):
        super(TransformerDecoder, self).__init__()
        self.tok_embedding = nn.Embedding(output_dim, d_model, padding_idx=PAD_token)
        self.pos_embedding = PositionalEncoding(d_model, dropout_p, max_len)
        self.layers = nn.ModuleList([TransformerDecoderLayer(d_model, n_heads, d_ff, dropout_p)
                                     for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, output_dim)
        self.dropout = nn.Dropout(p=dropout_p)
        self.scale = torch.sqrt(torch.FloatTensor([d_model])).to(device)
        
    def forward(self, tgt, enc_src, tgt_mask, src_mask):
        tgt_emb = self.tok_embedding(tgt) * self.scale
        tgt_pos = self.pos_embedding(tgt_emb.transpose(0, 1))
        tgt = self.dropout(tgt_pos.transpose(0, 1))
        for layer in self.layers:
            tgt, attention = layer(tgt, enc_src, tgt_mask, src_mask)
        output = self.fc_out(tgt)
        return output, attention

class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, tgt_pad_idx, device):
        super(Transformer, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx
        self.device = device
        
        # --- OPTIMIZATION: WEIGHT TYING ---
        self.decoder.fc_out.weight = self.decoder.tok_embedding.weight
        print("Applied Weight Tying between Decoder Embedding and Final FC Layer.")
        
    def make_src_mask(self, src):
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask
    
    def make_tgt_mask(self, tgt):
        tgt_pad_mask = (tgt != self.tgt_pad_idx).unsqueeze(1).unsqueeze(2)
        tgt_seq_len = tgt.shape[1]
        tgt_sub_mask = torch.tril(torch.ones((tgt_seq_len, tgt_seq_len), device=self.device)).bool()
        tgt_mask = tgt_pad_mask & tgt_sub_mask
        return tgt_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_src = self.encoder(src, src_mask)
        output, attention = self.decoder(tgt, enc_src, tgt_mask, src_mask)
        return output

## Step 7.5: Custom LR Scheduler (Noam)

In [8]:
class NoamOpt:
    "Optimizer wrapper that implements the Transformer LR schedule."
    def __init__(self, d_model, factor, warmup, optimizer):
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.d_model = d_model
        self._rate = 0
        
    def step(self):
        "Update parameters and rate"
        self._step += 1
        rate = self.rate()
        for p in self.optimizer.param_groups:
            p['lr'] = rate
        self._rate = rate
        self.optimizer.step()
        
    def rate(self, step = None):
        "Implement the lrate formula"
        if step is None:
            step = self._step
        if step == 0:
            return 0
        return self.factor * \
            (self.d_model ** (-0.5) *
            min(step ** (-0.5), step * self.warmup ** (-1.5)))
    
    def zero_grad(self):
        self.optimizer.zero_grad()
        
def get_std_opt(model, d_model, warmup, factor=1):
    "Helper function to create the Noam optimizer"
    return NoamOpt(d_model, factor, warmup,
            torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))

# -----------------------------------------------------
# ----------------- ENGLISH-BENGALI RUN -----------------
# -----------------------------------------------------

## Step 4 (Bengali): BPE Tokenization and Data Prep

In [9]:
LANG_PAIR_KEY_B = "English-Bengali"
OUTPUT_CSV_NAME_B = "answersB.csv"
BEST_MODEL_NAME_B = "best-model-bengali-transformer.pt"

# --- 1. Extract Data ---
print(f"Extracting Bengali data...")
source_sentences_train_b, target_sentences_train_b, id_train_b = extract_data(train_data, LANG_PAIR_KEY_B, "Train")
source_sentences_val_b, _, id_val_b = extract_data(val_data, LANG_PAIR_KEY_B, "Validation")
source_sentences_test_b, _, id_test_b = extract_data(test_data, LANG_PAIR_KEY_B, "Test")

print("\n--- Bengali Data Loading Summary ---")
print(f"Train examples: {len(source_sentences_train_b)}")
print(f"Valid examples: {len(source_sentences_val_b)}")
print(f"Test examples: {len(source_sentences_test_b)}")

# --- 2. Train BPE Tokenizer ---
print("\nPreparing text for BPE training...")
# Combine all training sentences (source and target) to train the tokenizer
all_text_b = "\n".join(source_sentences_train_b + target_sentences_train_b)
vocab_list_b, tokenizer_b = train_bpe_tokenizer(all_text_b, vocab_size=BPE_VOCAB_SIZE)
bpe_vocab_b = BPEVocab(vocab_list_b)

print("\n--- Bengali BPE Vocab Summary ---")
print(f"BPE Vocab Size: {bpe_vocab_b.n_words}")

# --- 3. Tokenize All Datasets with BPE ---
print("\nTokenizing all Bengali datasets with BPE...")
en_train_tokens_b = [bpe_preprocess_and_tokenize(s, tokenizer_b) for s in tqdm(source_sentences_train_b, desc="Tokenizing en_train_b")]
tgt_train_tokens_b = [bpe_preprocess_and_tokenize(s, tokenizer_b) for s in tqdm(target_sentences_train_b, desc="Tokenizing tgt_train_b")]
en_val_tokens_b = [bpe_preprocess_and_tokenize(s, tokenizer_b) for s in tqdm(source_sentences_val_b, desc="Tokenizing en_val_b")]
en_test_tokens_b = [bpe_preprocess_and_tokenize(s, tokenizer_b) for s in tqdm(source_sentences_test_b, desc="Tokenizing en_test_b")]

# --- 4. Encode and Pad All Datasets ---
print("\nEncoding and padding Bengali datasets...")
en_train_encoded_b = [encode_and_pad_bpe(s, bpe_vocab_b, MAX_LENGTH) for s in tqdm(en_train_tokens_b, desc="Encoding en_train_b")]
tgt_train_encoded_b = [encode_and_pad_bpe(s, bpe_vocab_b, MAX_LENGTH) for s in tqdm(tgt_train_tokens_b, desc="Encoding tgt_train_b")]
en_val_encoded_b = [encode_and_pad_bpe(s, bpe_vocab_b, MAX_LENGTH) for s in tqdm(en_val_tokens_b, desc="Encoding en_val_b")]
en_test_encoded_b = [encode_and_pad_bpe(s, bpe_vocab_b, MAX_LENGTH) for s in tqdm(en_test_tokens_b, desc="Encoding en_test_b")]

# --- 5. Create DataLoaders ---
BATCH_SIZE = 64
en_train_tensor_b = torch.LongTensor(en_train_encoded_b).to(device)
tgt_train_tensor_b = torch.LongTensor(tgt_train_encoded_b).to(device)
en_val_tensor_b = torch.LongTensor(en_val_encoded_b).to(device)
en_test_tensor_b = torch.LongTensor(en_test_encoded_b).to(device)

train_ds_b = TensorDataset(en_train_tensor_b, tgt_train_tensor_b)
val_pred_ds_b = TensorDataset(en_val_tensor_b) 
test_pred_ds_b = TensorDataset(en_test_tensor_b) 

train_loader_b = DataLoader(train_ds_b, shuffle=True, batch_size=BATCH_SIZE)
val_pred_loader_b = DataLoader(val_pred_ds_b, shuffle=False, batch_size=1) 
test_pred_loader_b = DataLoader(test_pred_ds_b, shuffle=False, batch_size=1) 

print(f"\nBengali Train DataLoader: {len(train_loader_b)} batches of size {BATCH_SIZE}")
print(f"Bengali Valid Inference Loader: {len(val_pred_loader_b)} batches of size 1")
print(f"Bengali Test Inference Loader: {len(test_pred_loader_b)} batches of size 1")

Extracting Bengali data...

--- Bengali Data Loading Summary ---
Train examples: 68849
Valid examples: 9836
Test examples: 19672

Preparing text for BPE training...
Starting BPE training with vocab size 15000...
BPE: Prepared 2140891 words for training.
BPE: Counting initial bigrams...
BPE: Starting merge loop...


100%|██████████| 14739/14739 [06:56<00:00, 35.41it/s] 


BPE: Training complete. Final vocab size: 15000

--- Bengali BPE Vocab Summary ---
BPE Vocab Size: 15000

Tokenizing all Bengali datasets with BPE...


Tokenizing en_test_b: 100%|██████████| 19672/19672 [01:01<00:00, 320.81it/s]



Encoding and padding Bengali datasets...


Encoding en_test_b: 100%|██████████| 19672/19672 [00:00<00:00, 185422.76it/s]



Bengali Train DataLoader: 1076 batches of size 64
Bengali Valid Inference Loader: 9836 batches of size 1
Bengali Test Inference Loader: 19672 batches of size 1


## Step 8 (Bengali): Train the Model

In [10]:
def train_fn_transformer(model, loader, optimizer, criterion, clip):
    """Performs one epoch of training for the Transformer."""
    model.train()
    epoch_loss = 0
    
    for batch in tqdm(loader, desc="Training"):
        src, tgt = batch
        src, tgt = src.to(device), tgt.to(device)
        
        optimizer.zero_grad()
        
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        
        output = model(src, tgt_input)
        
        output_dim = output.shape[-1]
        output_for_loss = output.reshape(-1, output_dim)
        tgt_for_loss = tgt_output.reshape(-1)
        
        loss = criterion(output_for_loss, tgt_for_loss)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
        
        optimizer.step()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(loader)

In [11]:
# -- Hyperparameters --
INPUT_DIM_B = bpe_vocab_b.n_words
OUTPUT_DIM_B = bpe_vocab_b.n_words # BPE shares vocab
D_MODEL = 256     # Embedding dimension
N_HEADS = 8       # Number of attention heads
D_FF = 512        # Dimension of the feed-forward layer
N_LAYERS = 3      # Number of Encoder/Decoder layers
DROPOUT = 0.1
NUM_EPOCHS = FINAL_EPOCH # Use the 30-epoch variable
CLIP = 1.0
WARMUP_STEPS_B = 4000 

# -- Model Instantiation --
encoder_b = TransformerEncoder(INPUT_DIM_B, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)
decoder_b = TransformerDecoder(OUTPUT_DIM_B, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)

model_b = Transformer(encoder_b, decoder_b, bpe_vocab_b.pad_idx, bpe_vocab_b.pad_idx, device).to(device)

# Initialize weights
def initialize_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)
model_b.apply(initialize_weights)

# --- Use the Noam Optimizer wrapper ---
optimizer_b = get_std_opt(model_b, D_MODEL, WARMUP_STEPS_B)

criterion_b = nn.CrossEntropyLoss(ignore_index=bpe_vocab_b.pad_idx)

print("--- Bengali Transformer Model Summary (with BPE) ---")
print(f"Source/Target BPE Vocab: {INPUT_DIM_B}")
print(f"D_MODEL: {D_MODEL}, N_HEADS: {N_HEADS}, N_LAYERS: {N_LAYERS}")

Applied Weight Tying between Decoder Embedding and Final FC Layer.
--- Bengali Transformer Model Summary (with BPE) ---
Source/Target BPE Vocab: 15000
D_MODEL: 256, N_HEADS: 8, N_LAYERS: 3


In [12]:
print("Starting Bengali Model Training...")

train_losses_b = []

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    train_loss = train_fn_transformer(model_b, train_loader_b, optimizer_b, criterion_b, CLIP)
    train_losses_b.append(train_loss)
    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    epoch_num = epoch + 1
    
    # Save a checkpoint if it's in our list
    if epoch_num in EPOCH_CHECKPOINTS:
        if epoch_num == FINAL_EPOCH:
            save_path = BEST_MODEL_NAME_B
        else:
            save_path = f"best-model-bengali-transformer-bpe_{epoch_num}.pt"
        
        torch.save(model_b.state_dict(), save_path)
        print(f"\n--- Saved model checkpoint: {save_path} ---")
    
    print(f'\nEpoch: {epoch_num:02} | Time: {epoch_mins}m {epoch_secs}s | LR: {optimizer_b.rate():.6f}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: N/A (No targets)')

print("Bengali Training finished.")

Starting Bengali Model Training...


Training: 100%|██████████| 1076/1076 [01:40<00:00, 10.70it/s]



Epoch: 01 | Time: 1m 40s | LR: 0.000266
	Train Loss: 5.869 | Train PPL: 354.024
	 Val. Loss: N/A (No targets)


Training: 100%|██████████| 1076/1076 [01:40<00:00, 10.70it/s]



Epoch: 02 | Time: 1m 40s | LR: 0.000532
	Train Loss: 4.205 | Train PPL:  67.043
	 Val. Loss: N/A (No targets)


...


Epoch: 28 | Time: 1m 40s | LR: 0.000360
	Train Loss: 1.799 | Train PPL:   6.043
	 Val. Loss: N/A (No targets)


Training: 100%|██████████| 1076/1076 [01:39<00:00, 10.77it/s]



Epoch: 29 | Time: 1m 39s | LR: 0.000354
	Train Loss: 1.783 | Train PPL:   5.950
	 Val. Loss: N/A (No targets)


Training: 100%|██████████| 1076/1076 [01:40<00:00, 10.74it/s]



--- Saved model checkpoint: best-model-bengali-transformer.pt ---

Epoch: 30 | Time: 1m 40s | LR: 0.000348
	Train Loss: 1.769 | Train PPL:   5.864
	 Val. Loss: N/A (No targets)
Bengali Training finished.


## Step 10 & 11 (Bengali): Inference and Save Checkpoints

We now loop through our saved checkpoints (10, 20, 30), load each one, generate predictions **for the test set**, and save the corresponding CSV.

In [13]:
def translate_sentence_transformer(sentence_str, tokenizer, vocab, model, device, max_length=MAX_LENGTH):
    """Translates a single raw string using BPE and the Transformer."""
    model.eval()
    
    # 1. Tokenize string with BPE
    sentence_tokens = bpe_preprocess_and_tokenize(sentence_str, tokenizer)
    
    # 2. Encode and pad tokens
    encoded_sentence = encode_and_pad_bpe(sentence_tokens, vocab, max_length)
    src_tensor = torch.LongTensor(encoded_sentence).unsqueeze(0).to(device)
    
    # 3. Create source mask
    src_mask = model.make_src_mask(src_tensor)
    
    # 4. Run encoder ONCE
    with torch.no_grad():
        enc_src = model.encoder(src_tensor, src_mask)
    
    # 5. Start decoder loop with <SOS> token
    tgt_indices = [vocab.sos_idx]
    
    for _ in range(max_length):
        tgt_tensor = torch.LongTensor(tgt_indices).unsqueeze(0).to(device)
        
        # 6. Create target mask (for current translated sequence)
        tgt_mask = model.make_tgt_mask(tgt_tensor)
        
        # 7. Decoder forward pass
        with torch.no_grad():
            output, attention = model.decoder(tgt_tensor, enc_src, tgt_mask, src_mask)
        
        # 8. Get the most likely token from the *last* time step
        pred_token_idx = output.argmax(2)[:, -1].item()
        tgt_indices.append(pred_token_idx)
        
        # 9. If <EOS>, stop translating
        if pred_token_idx == vocab.eos_idx:
            break
            
    # 10. Convert indices back to tokens (skipping <SOS>)
    decoded_tokens = [vocab.index2word[i] for i in tgt_indices[1:] if i != vocab.eos_idx]
    
    # 11. Detokenize BPE tokens back to a string
    return detokenize_bpe(decoded_tokens)

In [14]:
print("\n--- Starting Bengali Inference for all checkpoints ---")

for epoch_num in EPOCH_CHECKPOINTS:
    if epoch_num == FINAL_EPOCH:
        model_path = BEST_MODEL_NAME_B
        csv_path = OUTPUT_CSV_NAME_B
    else:
        model_path = f"best-model-bengali-transformer-bpe_{epoch_num}.pt"
        csv_path = f"answersB_{epoch_num}.csv"
    
    print(f"\n--- Generating TEST SET predictions for epoch {epoch_num} ---")
    print(f"Loading model: {model_path}")
    
    # Must re-create the model to load the state dict
    encoder_b = TransformerEncoder(INPUT_DIM_B, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)
    decoder_b = TransformerDecoder(OUTPUT_DIM_B, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)
    model_b = Transformer(encoder_b, decoder_b, bpe_vocab_b.pad_idx, bpe_vocab_b.pad_idx, device).to(device)

    try:
        model_b.load_state_dict(torch.load(model_path))
    except FileNotFoundError:
        print(f"WARNING: Model file {model_path} not found. Skipping this epoch.")
        continue
    
    test_translations_b = []
    print(f"Generating {len(source_sentences_test_b)} translations for TEST set...")
    # Note: We iterate over the *raw source sentences* for BPE
    for sentence_str in tqdm(source_sentences_test_b):
        translation = translate_sentence_transformer(sentence_str, tokenizer_b, bpe_vocab_b, model_b, device, MAX_LENGTH)
        test_translations_b.append(translation)
    
    # --- Save TEST SET Predictions --- 
    df_submission_test_b = pd.DataFrame()
    df_submission_test_b["ID"] = id_test_b
    df_submission_test_b["Translation"] = test_translations_b
    df_submission_test_b.to_csv(csv_path, index=False)
    print(f"TEST set submission file saved as {csv_path}")

print("\n--- Bengali inference complete. ---")

# --- Free up memory --- 
del model_b, encoder_b, decoder_b, optimizer_b, criterion_b, train_loader_b
del en_train_tensor_b, tgt_train_tensor_b, en_val_tensor_b, en_test_tensor_b
torch.cuda.empty_cache()
print("Cleaned up Bengali model and data memory.")


--- Starting Bengali Inference for all checkpoints ---

--- Generating TEST SET predictions for epoch 10 ---
Loading model: best-model-bengali-transformer-bpe_10.pt
Applied Weight Tying between Decoder Embedding and Final FC Layer.
Generating 19672 translations for TEST set...


100%|██████████| 19672/19672 [56:36<00:00,  5.79it/s]


TEST set submission file saved as answersB_10.csv

--- Generating TEST SET predictions for epoch 20 ---
Loading model: best-model-bengali-transformer-bpe_20.pt
Applied Weight Tying between Decoder Embedding and Final FC Layer.
Generating 19672 translations for TEST set...


100%|██████████| 19672/19672 [55:03<00:00,  5.96it/s]


TEST set submission file saved as answersB_20.csv

--- Generating TEST SET predictions for epoch 30 ---
Loading model: best-model-bengali-transformer.pt
Applied Weight Tying between Decoder Embedding and Final FC Layer.
Generating 19672 translations for TEST set...


100%|██████████| 19672/19672 [55:19<00:00,  5.93it/s]


TEST set submission file saved as answersB.csv

--- Bengali inference complete. ---
Cleaned up Bengali model and data memory.


# -----------------------------------------------------
# ----------------- ENGLISH-HINDI RUN -------------------
# -----------------------------------------------------

## Step 4 (Hindi): BPE Tokenization and Data Prep

In [15]:
LANG_PAIR_KEY_H = "English-Hindi"
OUTPUT_CSV_NAME_H = "answersH.csv"
BEST_MODEL_NAME_H = "best-model-hindi-transformer.pt"

# --- 1. Extract Data ---
print(f"Extracting Hindi data...")
source_sentences_train_h, target_sentences_train_h, id_train_h = extract_data(train_data, LANG_PAIR_KEY_H, "Train")
source_sentences_val_h, _, id_val_h = extract_data(val_data, LANG_PAIR_KEY_H, "Validation")
source_sentences_test_h, _, id_test_h = extract_data(test_data, LANG_PAIR_KEY_H, "Test")

print("\n--- Hindi Data Loading Summary ---")
print(f"Train examples: {len(source_sentences_train_h)}")
print(f"Valid examples: {len(source_sentences_val_h)}")
print(f"Test examples: {len(source_sentences_test_h)}")

# --- 2. Train BPE Tokenizer ---
print("\nPreparing text for BPE training...")
all_text_h = "\n".join(source_sentences_train_h + target_sentences_train_h)
vocab_list_h, tokenizer_h = train_bpe_tokenizer(all_text_h, vocab_size=BPE_VOCAB_SIZE)
bpe_vocab_h = BPEVocab(vocab_list_h)

print("\n--- Hindi BPE Vocab Summary ---")
print(f"BPE Vocab Size: {bpe_vocab_h.n_words}")

# --- 3. Tokenize All Datasets with BPE ---
print("\nTokenizing all Hindi datasets with BPE...")
en_train_tokens_h = [bpe_preprocess_and_tokenize(s, tokenizer_h) for s in tqdm(source_sentences_train_h, desc="Tokenizing en_train_h")]
tgt_train_tokens_h = [bpe_preprocess_and_tokenize(s, tokenizer_h) for s in tqdm(target_sentences_train_h, desc="Tokenizing tgt_train_h")]
en_val_tokens_h = [bpe_preprocess_and_tokenize(s, tokenizer_h) for s in tqdm(source_sentences_val_h, desc="Tokenizing en_val_h")]
en_test_tokens_h = [bpe_preprocess_and_tokenize(s, tokenizer_h) for s in tqdm(source_sentences_test_h, desc="Tokenizing en_test_h")]

# --- 4. Encode and Pad All Datasets ---
print("\nEncoding and padding Hindi datasets...")
en_train_encoded_h = [encode_and_pad_bpe(s, bpe_vocab_h, MAX_LENGTH) for s in tqdm(en_train_tokens_h, desc="Encoding en_train_h")]
tgt_train_encoded_h = [encode_and_pad_bpe(s, bpe_vocab_h, MAX_LENGTH) for s in tqdm(tgt_train_tokens_h, desc="Encoding tgt_train_h")]
en_val_encoded_h = [encode_and_pad_bpe(s, bpe_vocab_h, MAX_LENGTH) for s in tqdm(en_val_tokens_h, desc="Encoding en_val_h")]
en_test_encoded_h = [encode_and_pad_bpe(s, bpe_vocab_h, MAX_LENGTH) for s in tqdm(en_test_tokens_h, desc="Encoding en_test_h")]

# --- 5. Create DataLoaders ---
# BATCH_SIZE is already defined
en_train_tensor_h = torch.LongTensor(en_train_encoded_h).to(device)
tgt_train_tensor_h = torch.LongTensor(tgt_train_encoded_h).to(device)
en_val_tensor_h = torch.LongTensor(en_val_encoded_h).to(device)
en_test_tensor_h = torch.LongTensor(en_test_encoded_h).to(device)

train_ds_h = TensorDataset(en_train_tensor_h, tgt_train_tensor_h)
val_pred_ds_h = TensorDataset(en_val_tensor_h)
test_pred_ds_h = TensorDataset(en_test_tensor_h) 

train_loader_h = DataLoader(train_ds_h, shuffle=True, batch_size=BATCH_SIZE)
val_pred_loader_h = DataLoader(val_pred_ds_h, shuffle=False, batch_size=1)
test_pred_loader_h = DataLoader(test_pred_ds_h, shuffle=False, batch_size=1)

print(f"\nHindi Train DataLoader: {len(train_loader_h)} batches of size {BATCH_SIZE}")
print(f"Hindi Valid Inference Loader: {len(val_pred_loader_h)} batches of size 1")
print(f"Hindi Test Inference Loader: {len(test_pred_loader_h)} batches of size 1")

Extracting Hindi data...

--- Hindi Data Loading Summary ---
Train examples: 80797
Valid examples: 11543
Test examples: 23085

Preparing text for BPE training...
Starting BPE training with vocab size 15000...
BPE: Prepared 2908647 words for training.
BPE: Counting initial bigrams...
BPE: Starting merge loop...


100%|██████████| 14739/14739 [08:33<00:00, 28.71it/s] 


BPE: Training complete. Final vocab size: 15000

--- Hindi BPE Vocab Summary ---
BPE Vocab Size: 15000

Tokenizing all Hindi datasets with BPE...


Tokenizing en_test_h: 100%|██████████| 23085/23085 [01:15<00:00, 304.13it/s]



Encoding and padding Hindi datasets...


Encoding en_test_h: 100%|██████████| 23085/23085 [00:00<00:00, 163927.10it/s]



Hindi Train DataLoader: 1263 batches of size 64
Hindi Valid Inference Loader: 11543 batches of size 1
Hindi Test Inference Loader: 23085 batches of size 1


## Step 8 (Hindi): Train the Model

In [16]:
# -- Hyperparameters --
INPUT_DIM_H = bpe_vocab_h.n_words
OUTPUT_DIM_H = bpe_vocab_h.n_words
D_MODEL = 256     # Embedding dimension
N_HEADS = 8       # Number of attention heads
D_FF = 512        # Dimension of the feed-forward layer
N_LAYERS = 3      # Number of Encoder/Decoder layers
DROPOUT = 0.1
NUM_EPOCHS = FINAL_EPOCH 
CLIP = 1.0
WARMUP_STEPS_H = 4000 

# -- Model Instantiation --
encoder_h = TransformerEncoder(INPUT_DIM_H, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)
decoder_h = TransformerDecoder(OUTPUT_DIM_H, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)

model_h = Transformer(encoder_h, decoder_h, bpe_vocab_h.pad_idx, bpe_vocab_h.pad_idx, device).to(device)

# Initialize weights
model_h.apply(initialize_weights)

# --- Use the Noam Optimizer wrapper ---
optimizer_h = get_std_opt(model_h, D_MODEL, WARMUP_STEPS_H)

criterion_h = nn.CrossEntropyLoss(ignore_index=bpe_vocab_h.pad_idx)

print("--- Hindi Transformer Model Summary (with BPE) ---")
print(f"Source/Target BPE Vocab: {INPUT_DIM_H}")
print(f"D_MODEL: {D_MODEL}, N_HEADS: {N_HEADS}, N_LAYERS: {N_LAYERS}")

Applied Weight Tying between Decoder Embedding and Final FC Layer.
--- Hindi Transformer Model Summary (with BPE) ---
Source/Target BPE Vocab: 15000
D_MODEL: 256, N_HEADS: 8, N_LAYERS: 3


In [17]:
print("Starting Hindi Model Training...")

train_losses_h = []

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    train_loss = train_fn_transformer(model_h, train_loader_h, optimizer_h, criterion_h, CLIP)
    train_losses_h.append(train_loss)
    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    epoch_num = epoch + 1
    
    # Save a checkpoint if it's in our list
    if epoch_num in EPOCH_CHECKPOINTS:
        if epoch_num == FINAL_EPOCH:
            save_path = BEST_MODEL_NAME_H
        else:
            save_path = f"best-model-hindi-transformer-bpe_{epoch_num}.pt"
        
        torch.save(model_h.state_dict(), save_path)
        print(f"\n--- Saved model checkpoint: {save_path} ---")
    
    print(f'\nEpoch: {epoch_num:02} | Time: {epoch_mins}m {epoch_secs}s | LR: {optimizer_h.rate():.6f}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: N/A (No targets)')

print("Hindi Training finished.")

Starting Hindi Model Training...


Training: 100%|██████████| 1263/1263 [01:58<00:00, 10.62it/s]



Epoch: 01 | Time: 1m 58s | LR: 0.000312
	Train Loss: 5.179 | Train PPL: 177.464
	 Val. Loss: N/A (No targets)


Training: 100%|██████████| 1263/1263 [01:58<00:00, 10.65it/s]



Epoch: 02 | Time: 1m 58s | LR: 0.000624
	Train Loss: 3.483 | Train PPL:  32.541
	 Val. Loss: N/A (No targets)


...


Epoch: 28 | Time: 1m 57s | LR: 0.000332
	Train Loss: 1.302 | Train PPL:   3.675
	 Val. Loss: N/A (No targets)


Training: 100%|██████████| 1263/1263 [01:57<00:00, 10.76it/s]



Epoch: 29 | Time: 1m 57s | LR: 0.000327
	Train Loss: 1.290 | Train PPL:   3.634
	 Val. Loss: N/A (No targets)


Training: 100%|██████████| 1263/1263 [01:57<00:00, 10.76it/s]


--- Saved model checkpoint: best-model-hindi-transformer.pt ---

Epoch: 30 | Time: 1m 57s | LR: 0.000321
	Train Loss: 1.281 | Train PPL:   3.599
	 Val. Loss: N/A (No targets)
Hindi Training finished.


## Step 10 & 11 (Hindi): Inference and Save Checkpoints

We now loop through our saved checkpoints for the HINDI model.

In [18]:
print("\n--- Starting Hindi Inference for all checkpoints ---")

for epoch_num in EPOCH_CHECKPOINTS:
    if epoch_num == FINAL_EPOCH:
        model_path = BEST_MODEL_NAME_H
        csv_path = OUTPUT_CSV_NAME_H
    else:
        model_path = f"best-model-hindi-transformer-bpe_{epoch_num}.pt"
        csv_path = f"answersH_{epoch_num}.csv"
    
    print(f"\n--- Generating TEST SET predictions for epoch {epoch_num} ---")
    print(f"Loading model: {model_path}")
    
    # Re-create model structure to load weights
    encoder_h = TransformerEncoder(INPUT_DIM_H, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)
    decoder_h = TransformerDecoder(OUTPUT_DIM_H, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT, MAX_LENGTH).to(device)
    model_h = Transformer(encoder_h, decoder_h, bpe_vocab_h.pad_idx, bpe_vocab_h.pad_idx, device).to(device)

    try:
        model_h.load_state_dict(torch.load(model_path))
    except FileNotFoundError:
        print(f"WARNING: Model file {model_path} not found. Skipping this epoch.")
        continue
    
    test_translations_h = []
    print(f"Generating {len(source_sentences_test_h)} translations for TEST set...")
    for sentence_str in tqdm(source_sentences_test_h):
        translation = translate_sentence_transformer(sentence_str, tokenizer_h, bpe_vocab_h, model_h, device, MAX_LENGTH)
        test_translations_h.append(translation)
    
    # --- Save TEST SET Predictions --- 
    df_submission_test_h = pd.DataFrame()
    df_submission_test_h["ID"] = id_test_h
    df_submission_test_h["Translation"] = test_translations_h
    df_submission_test_h.to_csv(csv_path, index=False)
    print(f"TEST set submission file saved as {csv_path}")

print("\n--- Hindi inference complete. ---_EPOCHS")


--- Starting Hindi Inference for all checkpoints ---

--- Generating TEST SET predictions for epoch 10 ---
Loading model: best-model-hindi-transformer-bpe_10.pt
Applied Weight Tying between Decoder Embedding and Final FC Layer.
Generating 23085 translations for TEST set...


100%|██████████| 23085/23085 [1:17:50<00:00,  4.94it/s]


TEST set submission file saved as answersH_10.csv

--- Generating TEST SET predictions for epoch 20 ---
Loading model: best-model-hindi-transformer-bpe_20.pt
Applied Weight Tying between Decoder Embedding and Final FC Layer.
Generating 23085 translations for TEST set...


100%|██████████| 23085/23085 [1:17:05<00:00,  4.99it/s]


TEST set submission file saved as answersH_20.csv

--- Generating TEST SET predictions for epoch 30 ---
Loading model: best-model-hindi-transformer.pt
Applied Weight Tying between Decoder Embedding and Final FC Layer.
Generating 23085 translations for TEST set...


100%|██████████| 23085/23085 [1:18:14<00:00,  4.92it/s]

TEST set submission file saved as answersH.csv

--- Hindi inference complete. ---_EPOCHS


## Final Step: Combine, Format, and Zip (All Epochs)

This final block will loop through all our checkpointed epochs (10, 20, 30) and create a separate `submission_X.zip` for each one.

In [3]:
import pandas as pd
import zipfile
print("\n--- Starting Final Combination and Zipping for All Epochs ---")

EPOCH_CHECKPOINTS = [10, 20, 30]
for epoch_num in EPOCH_CHECKPOINTS:
    print(f"\n--- Processing Epoch {epoch_num} ---")
    
    # # 1. Define filenames for this epoch
    # if epoch_num == FINAL_EPOCH:
    #     bengali_csv_path = "answersB.csv"
    #     hindi_csv_path = "answersH.csv"
    #     combined_csv_path = "answersBH.csv"
    #     final_answer_path = "answer.csv"
    #     zip_path = "submission.zip"
    # else:
    #     bengali_csv_path = f"answersB_{epoch_num}.csv"
    #     hindi_csv_path = f"answersH_{epoch_num}.csv"
    combined_csv_path = f"answersBH_{epoch_num}.csv"
    final_answer_path = f"answer_{epoch_num}.csv"
    zip_path = f"submission_{epoch_num}.zip"
        
    # 2. Combine B and H files
    try:
        df_bengali = pd.read_csv(r"/Users/dhruv/acads/CS779-IITK/Capstone Project/bpe transform opt kaggle run/results/answersB_10.csv")
        df_hindi = pd.read_csv(r"/Users/dhruv/acads/CS779-IITK/Capstone Project/bpe transform opt kaggle run/results/answersH_10.csv")
        df_combined = pd.concat([df_bengali, df_hindi])
        df_combined.to_csv(combined_csv_path, index=False)
        print(f"Combined {len(df_bengali)} B and {len(df_hindi)} H rows -> {combined_csv_path}")
    except FileNotFoundError as e:
        print(f"ERROR: Could not find input file {e}. Skipping epoch {epoch_num}.")
        continue
        
    # 3. Format into final answer.csv
    try:
        filtered_data = pd.read_csv(combined_csv_path)
        filtered_data['Translation'] = filtered_data['Translation'].fillna('')
        
        with open(final_answer_path, "w") as f:
            f.writelines("ID\tTranslation\n")
            for i, row in filtered_data.iterrows():
                # Ensure translation is a string and handle quotes
                translation_text = str(row["Translation"]).replace('"', '""')
                f.writelines(f'{row["ID"]}\t"{translation_text}"\n')
        print(f"Successfully created {final_answer_path}")
    except Exception as e:
         print(f"ERROR: Could not create {final_answer_path}. {e}")
         continue
    
    # 4. Zip the file
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            zf.write(final_answer_path)
        print(f"Successfully created {zip_path}!")
    except FileNotFoundError:
        print(f"ERROR: {final_answer_path} not found. Cannot create zip.")

print("\n--- All submissions created! ---_EPOCHS")


--- Starting Final Combination and Zipping for All Epochs ---

--- Processing Epoch 10 ---
Combined 19672 B and 23085 H rows -> answersBH_10.csv
Successfully created answer_10.csv
Successfully created submission_10.zip!

--- Processing Epoch 20 ---
Combined 19672 B and 23085 H rows -> answersBH_20.csv
Successfully created answer_20.csv
Successfully created submission_20.zip!

--- Processing Epoch 30 ---
Combined 19672 B and 23085 H rows -> answersBH_30.csv
Successfully created answer_30.csv
Successfully created submission_30.zip!

--- All submissions created! ---_EPOCHS


In [19]:
print("\n--- Starting Final Combination and Zipping for All Epochs ---")

for epoch_num in EPOCH_CHECKPOINTS:
    print(f"\n--- Processing Epoch {epoch_num} ---")
    
    # 1. Define filenames for this epoch
    if epoch_num == FINAL_EPOCH:
        bengali_csv_path = "answersB.csv"
        hindi_csv_path = "answersH.csv"
        combined_csv_path = "answersBH.csv"
        final_answer_path = "answer.csv"
        zip_path = "submission.zip"
    else:
        bengali_csv_path = f"answersB_{epoch_num}.csv"
        hindi_csv_path = f"answersH_{epoch_num}.csv"
        combined_csv_path = f"answersBH_{epoch_num}.csv"
        final_answer_path = f"answer_{epoch_num}.csv"
        zip_path = f"submission_{epoch_num}.zip"
        
    # 2. Combine B and H files
    try:
        df_bengali = pd.read_csv(bengali_csv_path)
        df_hindi = pd.read_csv(hindi_csv_path)
        df_combined = pd.concat([df_bengali, df_hindi])
        df_combined.to_csv(combined_csv_path, index=False)
        print(f"Combined {len(df_bengali)} B and {len(df_hindi)} H rows -> {combined_csv_path}")
    except FileNotFoundError as e:
        print(f"ERROR: Could not find input file {e}. Skipping epoch {epoch_num}.")
        continue
        
    # 3. Format into final answer.csv
    try:
        filtered_data = pd.read_csv(combined_csv_path)
        filtered_data['Translation'] = filtered_data['Translation'].fillna('')
        
        with open(final_answer_path, "w") as f:
            f.writelines("ID\tTranslation\n")
            for i, row in filtered_data.iterrows():
                # Ensure translation is a string and handle quotes
                translation_text = str(row["Translation"]).replace('"', '""')
                f.writelines(f'{row["ID"]}\t"{translation_text}"\n')
        print(f"Successfully created {final_answer_path}")
    except Exception as e:
         print(f"ERROR: Could not create {final_answer_path}. {e}")
         continue
    
    # 4. Zip the file
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            zf.write(final_answer_path)
        print(f"Successfully created {zip_path}!")
    except FileNotFoundError:
        print(f"ERROR: {final_answer_path} not found. Cannot create zip.")

print("\n--- All submissions created! ---_EPOCHS")


--- Starting Final Combination and Zipping for All Epochs ---

--- Processing Epoch 10 ---
Combined 19672 B and 23085 H rows -> answersBH_10.csv
Successfully created answer_10.csv
Successfully created submission_10.zip!

--- Processing Epoch 20 ---
Combined 19672 B and 23085 H rows -> answersBH_20.csv
Successfully created answer_20.csv
Successfully created submission_20.zip!

--- Processing Epoch 30 ---
Combined 19672 B and 23085 H rows -> answersBH.csv
Successfully created answer.csv
Successfully created submission.zip!

--- All submissions created! ---_EPOCHS


In [5]:
import pandas as pd
df_bengali = pd.read_csv(r"/Users/dhruv/acads/CS779-IITK/Capstone Project/bpe transform opt kaggle run/results/answersB_10.csv")
df_hindi = pd.read_csv(r"/Users/dhruv/acads/CS779-IITK/Capstone Project/bpe transform opt kaggle run/results/answersH_10.csv")
df_combined = pd.concat([df_bengali, df_hindi])
df_combined.shape

(42757, 2)

In [9]:
df_bengali.shape[0]

19672

In [10]:
df_hindi.shape[0]

23085

In [11]:
df_combined.head()

,ID,Translation
0,177039,বর্তমানঘটনাঘটেছে
1,177040,গোবিন্দবালাজীতারসাথেদেখাকরেনকিন্তুতাকেতারকাছেব...
2,177041,মধুবাধীরসাথেসাথেসাথেসাথেসাথেসাথেসাথেসাথেসাথেসা...
3,177042,যেহেতুসেতাকেজানতেপারেযেমায়েদেরমধ্যেমায়েদেরমধ্য...
4,177043,অস্ট্রেলিয়ারউল্লেখযোগ্যপরিমানকম।


In [12]:
df_combined.tail()

,ID,Translation
23080,563219,पिप्रेटकोएकईमेलभेजेंऔरपूछेंकिवहअबवहअबवहवहउसेमद...
23081,563220,बॉटमें’बोटो’में’जीनिया’प्रोफाइल’कीसरकारनेएकसंस...
23082,563221,v.../शुरुआतकेबादडा.फीडको4बजेस्टोरकरनेकेबादएकहल...
23083,563222,दोनोंहाथोंकोछालकेसाथछालकेदावादें।
23084,563223,हमारेपानीबिलबहुतऊँचाथा
